In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from src.config import PATHS
from src.merging import safe_merge, build_integrated_household_dataset, build_integrated_person_dataset

In [ ]:
df_h = pd.read_parquet(PATHS['processed'] / 'hogar_clean.parquet')
df_t = pd.read_parquet(PATHS['processed'] / 'tic_clean.parquet')
df_e = pd.read_parquet(PATHS['processed'] / 'educ_clean.parquet')
df_d = pd.read_parquet(PATHS['processed'] / 'diseno_clean.parquet')

print(f'Módulo Hogar : {df_h.shape[0]:,} filas × {df_h.shape[1]} columnas')
print(f'Módulo TIC (Personas) : {df_t.shape[0]:,} filas × {df_t.shape[1]} columnas')
print(f'Módulo Educación : {df_e.shape[0]:,} filas × {df_e.shape[1]} columnas')
print(f'Módulo Diseño Muestral: {df_d.shape[0]:,} filas × {df_d.shape[1]} columnas')

## 5.2 Hallazgo Crucial de Auditoría en Claves del DANE

## 5.3 Integración a Nivel Hogar (`data/analytical/analytic_hogar.parquet`)

In [ ]:
df_hh_int = build_integrated_household_dataset(df_h, df_d, df_t, df_e)

# Guardar en data/analytical/
out_h = PATHS['analytical'] / 'analytic_hogar.parquet'
df_hh_int.to_parquet(out_h, index=False)
print(f'[OK] Guardado: {out_h.name} ({df_hh_int.shape[0]:,} filas × {df_hh_int.shape[1]} columnas)')

## 5.4 Integración a Nivel Persona (`data/analytical/analytic_persona.parquet`)

In [ ]:
df_p_int = build_integrated_person_dataset(df_t, df_e, df_h, df_d)

# Guardar en data/analytical/
out_p = PATHS['analytical'] / 'analytic_persona.parquet'
df_p_int.to_parquet(out_p, index=False)
print(f'[OK] Guardado: {out_p.name} ({df_p_int.shape[0]:,} filas × {df_p_int.shape[1]} columnas)')

## 5.5 Verificación de Integridad y Métricas Post-Merge

In [ ]:
print('='*60)
print('RESUMEN DE AUDITORÍA POST-INTEGRACIÓN')
print('='*60)
print(f'Hogares originales DANE : {df_h.shape[0]:,}')
print(f'Hogares dataset integrado : {df_hh_int.shape[0]:,} (Ratio: {df_hh_int.shape[0]/df_h.shape[0]:.4f})')
print(f'Personas originales TIC : {df_t.shape[0]:,}')
print(f'Personas dataset integrado: {df_p_int.shape[0]:,} (Ratio: {df_p_int.shape[0]/df_t.shape[0]:.4f})')
print('='*60)

assert df_hh_int['DIRECTORIO'].isna().sum() == 0
assert df_hh_int['FEX_C'].isna().sum() == 0
assert df_p_int['ORDEN'].isna().sum() == 0